In [ ]:
%%capture
!pip install keras-cv

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow" # other options: tensorflow or torch
import json
import keras_cv
import keras
from keras import ops
import tensorflow as tf

import cv2
import pandas as pd
import numpy as np
from glob import glob
from tqdm.notebook import tqdm
import joblib

import matplotlib.pyplot as plt

In [ ]:
class CFG:
    verbose = 1  # Verbosity
    seed = 47  # Random seed
    neg_sample = 0.01 # Downsample negative calss
    pos_sample = 5.0  # Upsample positive class
    preset = "efficientnetv2_b2_imagenet"  # Name of pretrained classifier
    image_size = [128, 128]  # Input image size
    epochs = 10 # Training epochs
    batch_size = 256  # Batch size
    lr_mode = "cos" # LR scheduler mode from one of "cos", "step", "exp"
    class_names = ['target']
    num_classes = 1

keras.utils.set_random_seed(CFG.seed)

In [ ]:
ROOT = str(os.getcwd())
BASE_PATH = f"{ROOT}/drive/MyDrive/WAIT"
DATA_PATH = f"{BASE_PATH}/data/isic"
MODEL_PATH = f"{BASE_PATH}/models/enhanced/"
PROC_PATH = f"{MODEL_PATH}/enhanced/preprocessed_dataset/"

In [ ]:
# Train + Valid
df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_train_metadata_clean.parquet')

# Testing
testing_df = pd.read_parquet(f'{DATA_PATH}/enhanced/enhanced_test_metadata_clean.parquet')

In [ ]:
print("Class Distribution Before Sampling (%):")
display(df.target.value_counts(normalize=True)*100)

# Sampling
positive_df = df.query("target==0").sample(frac=CFG.neg_sample, random_state=CFG.seed)
negative_df = df.query("target==1").sample(frac=CFG.pos_sample, replace=True, random_state=CFG.seed)
df = pd.concat([positive_df, negative_df], axis=0).sample(frac=1.0)

print("\nCalss Distribution After Sampling (%):")
display(df.target.value_counts(normalize=True)*100)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(df['target']), y=df['target'])
class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

In [ ]:
import h5py

training_validation_hdf5 = h5py.File(f"{DATA_PATH}/train-image.hdf5", 'r')
testing_hdf5 = h5py.File(f"{DATA_PATH}/test-image.hdf5", 'r')

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

df = df.reset_index(drop=True) # ensure continuous index
df["fold"] = -1
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=CFG.seed)
for i, (training_idx, validation_idx) in enumerate(sgkf.split(df, y=df.target, groups=df.patient_id)):
    df.loc[validation_idx, "fold"] = int(i)

# Use first fold for training and validation
training_df = df.query("fold!=0")
validation_df = df.query("fold==0")
print(f"# Num Train: {len(training_df)} | Num Valid: {len(validation_df)}")

In [ ]:
# class distribution in training and validation
print("training:", training_df.target.value_counts(),
      "\nvalidation:", validation_df.target.value_counts(), sep='\n')

# 📊 | Tabular Features

### Enhanced


In [ ]:
# Categorical features which will be one hot encoded
CATEGORICAL_COLUMNS = ["sex", "anatom_site_general", "tbp_tile_type",
                       "tbp_lv_location", "attribution"]

NEW_CATEGORICAL_COLUMNS =["combined_anatomical_site"]

# Numerical features which will be normalized
NUMERIC_COLUMNS = ["age_approx", "tbp_lv_nevi_confidence", "clin_size_long_diam_mm",
           "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio", "tbp_lv_color_std_mean",
           "tbp_lv_deltaLBnorm", "tbp_lv_minorAxisMM"]

NEW_NUMERIC_COLUMNS = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast",
        "luminance_contrast", "lesion_color_difference", "border_complexity",
         "3d_position_distance", "perimeter_to_area_ratio",
        "lesion_visibility_score", "symmetry_border_consistency", "color_consistency",

        "size_age_interaction", "hue_color_std_interaction", "lesion_severity_index",
        "shape_complexity_index", "color_contrast_index", "log_lesion_area",
        "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast",
        "color_shape_composite_index", "3d_lesion_orientation", "overall_color_difference",
        "symmetry_perimeter_interaction", "comprehensive_lesion_index"]

# Tabular feature columns
FEAT_COLS = (CATEGORICAL_COLUMNS + NEW_CATEGORICAL_COLUMNS) + (NUMERIC_COLUMNS + NEW_NUMERIC_COLUMNS)

In [ ]:
def build_augmenter():
    # Define augmentations
    aug_layers = [
        keras_cv.layers.RandomCutout(height_factor=(0.02, 0.06), width_factor=(0.02, 0.06)),
        keras_cv.layers.RandomFlip(mode="horizontal"),
    ]

    # Apply augmentations to random samples
    aug_layers = [keras_cv.layers.RandomApply(x, rate=0.5) for x in aug_layers]

    # Build augmentation layer
    augmenter = keras_cv.layers.Augmenter(aug_layers)

    # Apply augmentations
    def augment(inp, label):
        images = inp["images"]
        aug_data = {"images": images}
        aug_data = augmenter(aug_data)
        inp["images"] = aug_data["images"]
        return inp, label
    return augment


def build_decoder(with_labels=True, target_size=CFG.image_size):
    def decode_image(inp):
        # Read jpeg image
        file_bytes = inp["images"]
        image = tf.io.decode_jpeg(file_bytes)

        # Resize
        image = tf.image.resize(image, size=target_size, method="area")

        # Rescale image
        image = tf.cast(image, tf.float32)
        image /= 255.0

        # Reshape
        image = tf.reshape(image, [*target_size, 3])

        inp["images"] = image
        return inp

    def decode_label(label, num_classes):
        label = tf.cast(label, tf.float32)
        label = tf.reshape(label, [num_classes])
        return label

    def decode_with_labels(inp, label=None):
        inp = decode_image(inp)
        label = decode_label(label, CFG.num_classes)
        return (inp, label)

    return decode_with_labels if with_labels else decode_image


def build_dataset(
    isic_ids,
    hdf5,
    features,
    labels=None,
    batch_size=32,
    decode_fn=None,
    augment_fn=None,
    augment=False,
    shuffle=1024,
    cache=True,
    drop_remainder=False,
):
    if decode_fn is None:
        decode_fn = build_decoder(labels is not None)

    if augment_fn is None:
        augment_fn = build_augmenter()

    AUTO = tf.data.experimental.AUTOTUNE

    images = [None]*len(isic_ids)
    for i, isic_id in enumerate(tqdm(isic_ids, desc="Loading Images ")):
        images[i] = hdf5[isic_id][()]

    inp = {"images": images, "features": features}
    slices = (inp, labels) if labels is not None else inp

    ds = tf.data.Dataset.from_tensor_slices(slices)
    ds = ds.cache() if cache else ds
    ds = ds.map(decode_fn, num_parallel_calls=AUTO)
    if shuffle:
        ds = ds.shuffle(shuffle, seed=CFG.seed)
        opt = tf.data.Options()
        opt.deterministic = False
        ds = ds.with_options(opt)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder)
    ds = ds.map(augment_fn, num_parallel_calls=AUTO) if augment else ds
    ds = ds.prefetch(AUTO)
    return ds

In [ ]:
# Build Training & Validation Dataset

print("# Training:")
training_features = dict(training_df[FEAT_COLS])
training_ids = training_df.isic_id.values
training_labels = training_df.target.values
training_ds = build_dataset(training_ids, training_validation_hdf5, training_features,
                         training_labels, batch_size=CFG.batch_size,
                         shuffle=True, augment=True)

print("# Validation:")
validation_features = dict(validation_df[FEAT_COLS])
validation_ids = validation_df.isic_id.values
validation_labels = validation_df.target.values
validation_ds = build_dataset(validation_ids, training_validation_hdf5, validation_features,
                         validation_labels, batch_size=CFG.batch_size,
                         shuffle=False, augment=False)


In [ ]:
feature_space = keras.utils.FeatureSpace(
    features={
        # Categorical features encoded as integers
        "sex": "string_categorical",
        "anatom_site_general": "string_categorical",
        "tbp_tile_type": "string_categorical",
        "tbp_lv_location": "string_categorical",
        "combined_anatomical_site": "string_categorical",
        "attribution": "string_categorical",
        # Numerical features to discretize
        "age_approx": "float_discretized",
        # Numerical features to normalize
        "tbp_lv_nevi_confidence": "float_normalized",
        "clin_size_long_diam_mm": "float_normalized",
        "tbp_lv_areaMM2": "float_normalized",
        "tbp_lv_area_perim_ratio": "float_normalized",
        "tbp_lv_color_std_mean": "float_normalized",
        "tbp_lv_deltaLBnorm": "float_normalized",
        "tbp_lv_minorAxisMM": "float_normalized",
        "lesion_size_ratio" : "float_normalized",
        "lesion_shape_index" : "float_normalized",
        "hue_contrast" : "float_normalized",
        "luminance_contrast": "float_normalized",
        "lesion_color_difference": "float_normalized",
        "border_complexity": "float_normalized",
        "3d_position_distance": "float_normalized",
        "perimeter_to_area_ratio": "float_normalized",
        "lesion_visibility_score": "float_normalized",
        "symmetry_border_consistency": "float_normalized",
        "color_consistency": "float_normalized",
        "size_age_interaction": "float_normalized",
        "hue_color_std_interaction": "float_normalized",
        "lesion_severity_index": "float_normalized",
        "shape_complexity_index": "float_normalized",
        "color_contrast_index": "float_normalized",
        "log_lesion_area": "float_normalized",
        "mean_hue_difference": "float_normalized",
        "std_dev_contrast": "float_normalized",
        "color_shape_composite_index": "float_normalized",
        "3d_lesion_orientation": "float_normalized",
        "overall_color_difference": "float_normalized",
        "symmetry_perimeter_interaction": "float_normalized",
        "comprehensive_lesion_index": "float_normalized",
        # numerical features already normalized
        "normalized_lesion_size": "float",
        },
    output_mode="concat",
)

In [ ]:
training_ds_with_no_labels = training_ds.map(lambda x, _: x["features"])
feature_space.adapt(training_ds_with_no_labels)
print(feature_space.get_encoded_features().shape)

In [ ]:
# fs_dir = os.path.join(MODEL_PATH, "model-metadata")
# os.makedirs(fs_dir, exist_ok=True)
# filepath = os.path.join(fs_dir, "feature_space_test.keras")

# feature_space.save(filepath)
# print(f"FeatureSpace saved at: '{filepath}'")

In [ ]:
# Apply Feature Processing

training_ds = training_ds.map(
    lambda x, y: ({"images": x["images"],
                   "features": feature_space(x["features"])}, y), num_parallel_calls=tf.data.AUTOTUNE)

validation_ds = validation_ds.map(
    lambda x, y: ({"images": x["images"],
                   "features": feature_space(x["features"])}, y), num_parallel_calls=tf.data.AUTOTUNE)

# Testing
print("# Testing:")
testing_features = dict(testing_df[FEAT_COLS])
testing_ids = testing_df.isic_id.values
testing_ds = build_dataset(testing_ids, testing_hdf5,
                        testing_features, batch_size=CFG.batch_size,
                         shuffle=False, augment=False, cache=False)

# Apply feature space processing
testing_ds = testing_ds.map(
    lambda x: {"images": x["images"],
               "features": feature_space(x["features"])}, num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
# loss & metric
auc = keras.metrics.AUC()
loss = keras.losses.BinaryCrossentropy(label_smoothing=0.02)

In [ ]:
# Define input layers
image_input = keras.Input(shape=(*CFG.image_size, 3), name="images")
feat_input = keras.Input(shape=(feature_space.get_encoded_features().shape[1],), name="features")
inp = {"images":image_input, "features":feat_input}

# Branch for image input
backbone = keras_cv.models.EfficientNetV2Backbone.from_preset(CFG.preset)
x1 = backbone(image_input)
x1 = keras.layers.GlobalAveragePooling2D()(x1)
x1 = keras.layers.Dropout(0.2)(x1)

# Branch for tabular/feature input
x2 = keras.layers.Dense(96, activation="selu")(feat_input)
x2 = keras.layers.Dense(128, activation="selu")(x2)
x2 = keras.layers.Dropout(0.1)(x2)

# Concatenate both branches
concat = keras.layers.Concatenate()([x1, x2])

# Output layer
out = keras.layers.Dense(1, activation="sigmoid", dtype="float32")(concat)

# Build model
model = keras.models.Model(inp, out)

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=loss,
    metrics=[auc],
)

# Model Summary
model.summary()

In [ ]:
# keras.utils.plot_model(model, show_shapes=True, show_layer_names=True, dpi=60)

In [ ]:
import math

def get_lr_callback(batch_size=8, mode='cos', epochs=10, plot=False):
    lr_start, lr_max, lr_min = 2.5e-5, 5e-6 * batch_size, 0.8e-5
    lr_ramp_ep, lr_sus_ep, lr_decay = 3, 0, 0.75

    def lrfn(epoch):  # Learning rate update function
        if epoch < lr_ramp_ep: lr = (lr_max - lr_start) / lr_ramp_ep * epoch + lr_start
        elif epoch < lr_ramp_ep + lr_sus_ep: lr = lr_max
        elif mode == 'exp': lr = (lr_max - lr_min) * lr_decay**(epoch - lr_ramp_ep - lr_sus_ep) + lr_min
        elif mode == 'step': lr = lr_max * lr_decay**((epoch - lr_ramp_ep - lr_sus_ep) // 2)
        elif mode == 'cos':
            decay_total_epochs, decay_epoch_index = epochs - lr_ramp_ep - lr_sus_ep + 3, epoch - lr_ramp_ep - lr_sus_ep
            phase = math.pi * decay_epoch_index / decay_total_epochs
            lr = (lr_max - lr_min) * 0.5 * (1 + math.cos(phase)) + lr_min
        return lr

    if plot:  # Plot lr curve if plot is True
        plt.figure(figsize=(10, 5))
        plt.plot(np.arange(epochs), [lrfn(epoch) for epoch in np.arange(epochs)], marker='o')
        plt.xlabel('epoch'); plt.ylabel('lr')
        plt.title('LR Scheduler')
        plt.show()

    return keras.callbacks.LearningRateScheduler(lrfn, verbose=False)  # Create lr callback

lr_cb = get_lr_callback(CFG.batch_size, mode="exp", plot=False)

In [ ]:
%%capture
!pip install --no-cache-dir tf-explain

In [ ]:
# tf.keras.backend.clear_session()

In [ ]:
import os
from pathlib import Path
from typing import Optional, Tuple, List, Union

import cv2
import numpy as np
import tensorflow as tf
from keras import Model


class MultiModalGradCAMEngine:
    """
    Grad-CAM engine for multimodal models (images + tabular features).

    Features:
    - Robust target selection: defaults to the 4D tensor feeding the first GlobalAveragePooling2D.
    - Two modes:
        * 'context'     → uses real tabular features.
        * 'image_only'  → replaces features with a baseline (zeros or provided vector).
    - Output:
        * grid: single grid image (HxWx3, float [0..1]) of horizontally stacked triptychs
        * triptychs: list of per-sample images (orig+pred | heatmap | overlay), each Hx(3W)x3 in [0..1]
        * preds: model predictions for the batch (numpy array)
    - Visualization:
        * Accepts a separate display_images batch or a deprocess_fn to convert model-input images to [0,1].
    """

    def __init__(
        self,
        use_guided_grads: bool = True,
        mode: str = 'context',              # 'context' or 'image_only'
        baseline: Optional[np.ndarray] = None,
        colormap: int = cv2.COLORMAP_JET,
        image_weight: float = 0.8,
        deprocess_fn: Optional[callable] = None,  # fn(img_float) -> float [0,1]
    ):
        assert mode in ('context', 'image_only'), "mode must be 'context' or 'image_only'"
        self.use_guided_grads = use_guided_grads
        self.mode = mode
        self.baseline = baseline
        self.colormap = colormap
        self.image_weight = float(image_weight)
        self.deprocess_fn = deprocess_fn

    def explain(
        self,
        validation_data: Union[
            Tuple[np.ndarray, np.ndarray],                      # (images, features)
            Tuple[dict, np.ndarray],                            # ({'images', 'features'}, labels)
            Tuple[np.ndarray, np.ndarray, np.ndarray],          # (images, features, labels)
            Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]  # (images, features, display_images, labels?)
        ],
        model: tf.keras.Model,
        class_index: int,
        layer_name: Optional[str] = None,
    ) -> Tuple[np.ndarray, List[np.ndarray], np.ndarray]:
        """
        Compute Grad-CAMs for a batch and return (grid, triptychs, preds).

        validation_data may be one of:
          - (images, features)
          - ({'images': images, 'features': features}, labels)
          - (images, features, labels)
          - (images, features, display_images[, labels])
        """
        # --- Unpack ---
        images, features, display_images = self._unpack(validation_data)

        # --- Build grad model ---
        conv_target_tensor = self._resolve_target_tensor(model, layer_name)
        grad_model = Model(inputs=model.inputs, outputs=[conv_target_tensor, model.output])

        # --- Build feeds in model.inputs order ---
        feeds = self._ordered_feeds(model, images, features)
        if self.mode == 'image_only':
            feeds = self._apply_baseline_to_feeds(model, feeds, images, features)

        # --- Forward + gradients ---
        with tf.GradientTape() as tape:
            conv_out, preds = grad_model(feeds, training=False)
            if preds.shape.rank == 2 and preds.shape[-1] == 1:
                target = preds[:, 0]           # binary sigmoid
            else:
                target = preds[:, class_index]  # multi-class or multi-label
        grads = tape.gradient(target, conv_out)
        if self.use_guided_grads:
            grads = tf.cast(conv_out > 0, tf.float32) * tf.cast(grads > 0, tf.float32) * grads

        # --- CAMs (B, Hc, Wc) ---
        cams = self._generate_cams(conv_out, grads)  # list of (Hc, Wc)
        cams = np.stack(cams, axis=0)

        # --- Triptychs with display images ---
        triptychs = []
        preds_np = preds.numpy() if isinstance(preds, tf.Tensor) else preds
        for i in range(images.shape[0]):
            pred_text = self._format_pred_text(preds_np[i])
            # Use provided display images if available; else deprocess model-input images for visualization
            vis_img = display_images[i] if display_images is not None else images[i]
            trip = self._compose_triptych(vis_img, cams[i], pred_text)
            triptychs.append(trip)

        grid = self._make_grid(triptychs)
        return grid, triptychs, preds_np

    # ------------------------ Saving helpers ------------------------
    def save_grid(self, grid: np.ndarray, output_dir: str, output_name: str) -> None:
        """Save the grid image to disk (RGB)."""
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        out_path = str(Path(output_dir) / output_name)
        img_u8 = np.uint8(255 * np.clip(grid, 0, 1))
        cv2.imwrite(out_path, cv2.cvtColor(img_u8, cv2.COLOR_RGB2BGR))

    def save_triptychs(
        self,
        triptychs: List[np.ndarray],
        base_dir: str,
        epoch_name: str,
        prefix: str = "sample",
    ) -> None:
        """
        Save each triptych as:
          base_dir/epoch_name/{prefix}_{i:03d}.png
        """
        out_dir = Path(base_dir) / epoch_name
        out_dir.mkdir(parents=True, exist_ok=True)
        for i, img in enumerate(triptychs):
            img_u8 = np.uint8(255 * np.clip(img, 0, 1))
            path = out_dir / f"{prefix}_{i:03d}.png"
            cv2.imwrite(str(path), cv2.cvtColor(img_u8, cv2.COLOR_RGB2BGR))

    # ------------------------ Internal helpers ------------------------
    def _make_grid(self, triptychs: List[np.ndarray]) -> np.ndarray:
        """Create a grid image; uses tf-explain grid_display if available, else a simple grid."""
        try:
            from tf_explain.utils.display import grid_display
            thumbs = np.stack([np.uint8(255 * np.clip(t, 0, 1)) for t in triptychs], axis=0)
            canvas = grid_display(thumbs)  # returns uint8 RGB
            return canvas.astype(np.float32) / 255.0
        except Exception:
            return self._naive_grid(triptychs)

    def _unpack(self, validation_data):
        """
        Accepts multiple shapes and returns (images, features, display_images or None).
        Supported:
          - (images, features)
          - ({'images','features'}, y)
          - (images, features, y)
          - (images, features, display_images[, y])
        """
        images = features = display_images = None
        if isinstance(validation_data, dict):
            images = validation_data['images']
            features = validation_data['features']
            return images, features, None
        if isinstance(validation_data, (list, tuple)):
            if len(validation_data) == 2:
                x, _ = validation_data
                if isinstance(x, dict):
                    return x['images'], x['features'], None
                if isinstance(x, (list, tuple)) and len(x) == 2:
                    return x[0], x[1], None
                return validation_data[0], validation_data[1], None
            if len(validation_data) == 3:
                a, b, c = validation_data
                # Treat third arg as display_images if it looks like an image batch (Tensor or ndarray)
                is_arr = isinstance(c, (np.ndarray, tf.Tensor))
                if is_arr:
                    try:
                        nd = c.ndim
                        ch = c.shape[-1]
                    except Exception:
                        nd, ch = None, None
                    if (nd is not None and nd >= 3 and ch in (1, 3)):
                        return a, b, c
                return a, b, None
            if len(validation_data) >= 4:
                return validation_data[0], validation_data[1], validation_data[2]
        raise ValueError("validation_data must contain images and features")

    def _resolve_target_tensor(self, model: tf.keras.Model, layer_name: Optional[str]):
        """
        Prefer the 4D tensor feeding the first GlobalAveragePooling2D.
        If layer_name is provided, use that layer's output; else fallback to last 4D layer.
        """
        if layer_name is not None:
            return model.get_layer(layer_name).output
        # Prefer the tensor feeding the first GAP
        for l in model.layers:
            if isinstance(l, tf.keras.layers.GlobalAveragePooling2D):
                return l.input
        # Fallback to last 4D tensor in the graph
        for l in reversed(model.layers):
            try:
                shp = l.output_shape
            except Exception:
                continue
            if isinstance(shp, tuple) and len(shp) == 4:
                return l.output
        raise ValueError("No suitable 4D conv tensor found for Grad-CAM target")

    def _ordered_feeds(self, model: tf.keras.Model, images, features):
        """Create a list of inputs respecting model.inputs order (rank-4 → images, rank-2 → features)."""
        feeds = []
        for kt in model.inputs:
            if len(kt.shape) == 4:
                feeds.append(images)
            elif len(kt.shape) == 2:
                feeds.append(features)
            else:
                raise ValueError(f"Unexpected input rank for {kt.name}: {kt.shape}")
        return feeds

    def _apply_baseline_to_feeds(self, model: tf.keras.Model, feeds, images, features):
        """Replace the features tensor in feeds with a baseline (zeros by default)."""
        feat_inp = next(inp for inp in model.inputs if len(inp.shape) == 2)
        F = int(feat_inp.shape[-1])
        if self.baseline is None:
            baseline = tf.zeros((tf.shape(images)[0], F), dtype=getattr(features, 'dtype', tf.float32))
        else:
            base = tf.convert_to_tensor(self.baseline)
            if base.shape.rank == 1:
                baseline = tf.broadcast_to(base[None, :], [tf.shape(images)[0], F])
            else:
                baseline = base
        feeds_out = []
        for kt, fed in zip(model.inputs, feeds):
            feeds_out.append(baseline if len(kt.shape) == 2 else fed)
        return feeds_out

    def _generate_cams(self, conv_outputs: tf.Tensor, grads: tf.Tensor) -> List[np.ndarray]:
        """Compute per-sample weighted sum of conv activations; returns list of 2D CAMs normalized to [0,1]."""
        cams = []
        conv_np = conv_outputs.numpy() if isinstance(conv_outputs, tf.Tensor) else conv_outputs
        grads_np = grads.numpy() if isinstance(grads, tf.Tensor) else grads
        B = conv_np.shape[0]
        for i in range(B):
            weights = grads_np[i].mean(axis=(0, 1), keepdims=False)  # (C,)
            cam = (conv_np[i] * weights[None, None, :]).sum(axis=-1)
            cam_min, cam_max = cam.min(), cam.max()
            cam = (cam - cam_min) / (cam_max - cam_min + 1e-8)
            cams.append(cam)
        return cams

    def _format_pred_text(self, pred_row):
        """Format prediction text for display: binary → p=..., multi-class → ŷ=k (p=...)."""
        try:
            arr = np.array(pred_row)
        except Exception:
            return f"p={float(pred_row):.3f}"
        if arr.ndim == 0:
            return f"p={float(arr):.3f}"
        arr = arr.ravel()
        if arr.size == 1:
            return f"p={float(arr[0]):.3f}"
        else:
            k = int(np.argmax(arr))
            p = float(arr[k])
            return f"ŷ={k} (p={p:.3f})"

    def _compose_triptych(self, image_like: np.ndarray, cam_2d: np.ndarray, pred_text: str) -> np.ndarray:
        """Create a 3-panel image (orig+text | heatmap | overlay), each panel HxWx3 in [0,1]."""
        img = self._to_display_rgb(image_like)
        H, W = img.shape[:2]
        # Heatmap
        heat = cv2.applyColorMap(np.uint8(255 * cv2.resize(cam_2d, (W, H))), self.colormap)
        heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB) / 255.0
        # Overlay
        overlay = self.image_weight * img + (1.0 - self.image_weight) * heat
        overlay = np.clip(overlay, 0, 1)
        # Original with prediction text
        orig = (img.copy() * 255).astype(np.uint8)
        txt = pred_text if isinstance(pred_text, str) else str(pred_text)
        cv2.putText(orig, txt, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(orig, txt, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 1, cv2.LINE_AA)
        orig = orig.astype(np.float32) / 255.0
        return np.hstack([orig, heat, overlay])

    # ---------------- Display helpers ----------------
    def _to_display_rgb(self, image: np.ndarray) -> np.ndarray:
        """
        Convert a model-input image to float RGB in [0,1] for visualization.
        Applies custom deprocess_fn if provided; else uses heuristics.
        """
        arr = image.numpy() if tf.is_tensor(image) else np.asarray(image)

        if self.deprocess_fn is not None:
            out = self.deprocess_fn(arr)
            return np.clip(self._ensure_rgb(out), 0.0, 1.0)

        # 1) uint8 → [0,1]
        if arr.dtype == np.uint8:
            arr = arr.astype(np.float32) / 255.0
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # 2) Float ranges
        arr = arr.astype(np.float32)
        amin, amax = float(arr.min()), float(arr.max())

        # true [-1,1] only if there are negatives
        if amin < 0.0 and amax <= 1.1:
            arr = (arr + 1.0) / 2.0
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # already [0,1]
        if 0.0 <= amin and amax <= 1.1:
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # [0,255] floats
        if amin >= -1.0 and amax > 1.5 and amax <= 260:
            arr = arr / 255.0
            return np.clip(self._ensure_rgb(arr), 0, 1)

        # fallback: per-image min-max
        rng = (amax - amin) if (amax - amin) > 1e-8 else 1.0
        arr = (arr - amin) / rng
        return np.clip(self._ensure_rgb(arr), 0, 1)

    def _ensure_rgb(self, arr: np.ndarray) -> np.ndarray:
        if arr.ndim == 2:
            arr = np.stack([arr, arr, arr], axis=-1)
        if arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)
        return arr

    def _naive_grid(self, triptychs: List[np.ndarray], cols: int = 2) -> np.ndarray:
        if not triptychs:
            return np.zeros((32, 32, 3), dtype=np.float32)
        H, W, C = triptychs[0].shape
        rows = int(np.ceil(len(triptychs) / cols))
        grid = np.ones((rows * H, cols * W, C), dtype=np.float32)
        for idx, img in enumerate(triptychs):
            r = idx // cols
            c = idx % cols
            grid[r*H:(r+1)*H, c*W:(c+1)*W] = np.clip(img, 0, 1)
        return grid

In [ ]:
import os
from pathlib import Path
from typing import Optional, Union, Tuple

import cv2
import numpy as np
import tensorflow as tf
from keras.callbacks import Callback

# NOTE: assumes MultiModalGradCAMEngine is already defined/imported in the notebook

class MultiModalGradCAMCallback(Callback):
    """
    Keras callback that runs MultiModalGradCAMEngine every epoch and:
      - logs a grid image to TensorBoard
      - saves per-sample triptychs to disk under base_dir/epoch_XXX/
      - (NEW) always saves at least one most-positive prediction from the frozen batch per epoch

    It freezes a fixed set of examples at on_train_begin so comparisons across epochs are meaningful.
    """

    def __init__(
        self,
        data_source: Union[tf.data.Dataset, Tuple, dict],
        class_index: int = 0,
        layer_name: Optional[str] = None,
        base_dir: str = "/tmp/gradcam",         # where triptychs are saved
        log_dir: Optional[str] = None,            # where TensorBoard images are logged; default: base_dir
        max_samples: int = 8,
        mode: str = "context",                   # 'context' or 'image_only'
        baseline: Optional[np.ndarray] = None,
        use_guided_grads: bool = True,
        colormap: int = cv2.COLORMAP_JET,
        image_weight: float = 0.8,
        deprocess_fn: Optional[callable] = None,  # if you don’t pass display_images
        grid_name: str = "gradcam_grid",         # TB tag
        file_prefix: str = "val",                # filename prefix for saved samples
        every_n_epochs: int = 1,                  # run every N epochs
        take_first_batch: bool = True,            # if Dataset: take its first batch
        dtype_cast: Optional[tf.dtypes.DType] = None,  # optional enforced dtype for inputs
        # --- Minimal isic24 switches ---
        ensure_positive: bool = True,             # NEW: always save the most-positive sample per epoch
        positive_threshold: float = 0.5,          # kept for potential use; argmax is saved regardless
    ):
        super().__init__()
        self.data_source = data_source
        self.class_index = class_index
        self.layer_name = layer_name
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)
        self.log_dir = str(self.base_dir if log_dir is None else log_dir)
        self.max_samples = int(max_samples)
        self.mode = mode
        self.baseline = baseline
        self.use_guided_grads = use_guided_grads
        self.colormap = colormap
        self.image_weight = float(image_weight)
        self.deprocess_fn = deprocess_fn
        self.grid_name = grid_name
        self.file_prefix = file_prefix
        self.every_n_epochs = max(1, int(every_n_epochs))
        self.take_first_batch = take_first_batch
        self.dtype_cast = dtype_cast
        # isic24 flags
        self.ensure_positive = bool(ensure_positive)
        self.positive_threshold = float(positive_threshold)

        # Frozen batch holders
        self._images = None
        self._features = None
        self._display_images = None

        self._writer = tf.summary.create_file_writer(self.log_dir)

    def _unpack_any(self, source):
        """Return (images, features, display_images or None) from various source shapes.
        Supported:
          - Dataset yielding (({'images','features'}, y) or {'images','features'})
          - Tuple/list: (images, features[, display_images[, y]]) OR ({'images','features'}, y)
          - dict: {'images': batch_images, 'features': batch_features}
        """
        if isinstance(source, tf.data.Dataset):
            batch = next(iter(source))
            x = batch[0] if isinstance(batch, (tuple, list)) else batch
            if isinstance(x, dict):
                return x.get('images'), x.get('features'), None
            if isinstance(x, (tuple, list)) and len(x) >= 2:
                return x[0], x[1], x[2] if len(x) >= 3 else None
            raise ValueError("Unsupported dataset element structure for Grad-CAM callback.")

        if isinstance(source, dict):
            return source['images'], source['features'], source.get('display_images', None)

        if isinstance(source, (tuple, list)):
            # ({'images','features'}, y)
            if len(source) == 2 and isinstance(source[0], dict):
                return source[0]['images'], source[0]['features'], None
            # (images, features[, display_images[, y]])
            if len(source) >= 2 and not isinstance(source[0], dict):
                images, features = source[0], source[1]
                display_images = None
                if len(source) >= 3:
                    c = source[2]
                    if isinstance(c, (np.ndarray, tf.Tensor)) and getattr(c, 'ndim', 0) >= 3:
                        display_images = c
                return images, features, display_images
        raise ValueError("`data_source` must be a Dataset, dict, or (images, features[, display_images]) tuple.")

    def on_train_begin(self, logs=None):
        """Freeze a fixed mini-batch for the whole training."""
        images, features, display_images = self._unpack_any(self.data_source)

        # clip to max_samples to protect memory
        n = min(self.max_samples, int(images.shape[0]))
        images = images[:n]
        features = features[:n]
        if display_images is not None:
            display_images = display_images[:n]

        # Optional dtype cast
        if self.dtype_cast is not None:
            images = tf.convert_to_tensor(images, dtype=self.dtype_cast)
            features = tf.convert_to_tensor(features, dtype=self.dtype_cast)
            if display_images is not None:
                display_images = tf.convert_to_tensor(display_images, dtype=tf.float32)

        self._images = images
        self._features = features
        self._display_images = display_images

    def on_epoch_end(self, epoch, logs=None):
        if (epoch % self.every_n_epochs) != 0:
            return
        if self._images is None or self._features is None:
            return

        # Instantiate engine each epoch to reflect current weights
        engine = MultiModalGradCAMEngine(
            use_guided_grads=self.use_guided_grads,
            mode=self.mode,
            baseline=self.baseline,
            colormap=self.colormap,
            image_weight=self.image_weight,
            deprocess_fn=self.deprocess_fn,
        )

        # Build data tuple expected by the engine
        if self._display_images is not None:
            val_data = (self._images, self._features, self._display_images)
        else:
            val_data = (self._images, self._features)

        # Explain on current model
        grid, triptychs, preds = engine.explain(
            validation_data=val_data,
            model=self.model,
            class_index=self.class_index,
            layer_name=self.layer_name,
        )

        # Log grid to TensorBoard
        with self._writer.as_default():
            grid_bhwc = np.expand_dims(np.clip(grid, 0, 1), 0)  # (1,H,W,3)
            tf.summary.image(self.grid_name, grid_bhwc, step=epoch)
            self._writer.flush()

        # Save triptychs to disk under epoch-named folder
        epoch_name = f"epoch_{epoch:03d}"
        engine.save_triptychs(triptychs, base_dir=str(self.base_dir), epoch_name=epoch_name, prefix=self.file_prefix)
        engine.save_grid(grid, output_dir=str(self.base_dir / epoch_name), output_name=f"{self.file_prefix}_grid.png")

        # --- NEW: Always save at least one most-positive sample per epoch ---
        if self.ensure_positive and len(triptychs) > 0:
            preds_arr = np.array(preds).reshape((len(triptychs), -1))
            # Binary (B,1) → probs; Multi-class (B,C) → class_index column
            if preds_arr.shape[1] == 1:
                probs = preds_arr[:, 0]
            else:
                probs = preds_arr[:, self.class_index]

            top_idx = int(np.argmax(probs))
            top_prob = float(probs[top_idx])

            # Save the corresponding triptych with probability in filename
            epoch_dir = self.base_dir / epoch_name
            epoch_dir.mkdir(parents=True, exist_ok=True)
            img_u8 = np.uint8(255 * np.clip(triptychs[top_idx], 0, 1))
            fname = f"pos_top_{top_idx:03d}_p{top_prob:.3f}.png"
            cv2.imwrite(str(epoch_dir / fname), cv2.cvtColor(img_u8, cv2.COLOR_RGB2BGR))

In [ ]:
XAI_CB_PATH = os.path.join(MODEL_PATH, 'xai', 'test', 'callback')
base_dir=os.path.join(XAI_CB_PATH, "images")  # PNGs here
log_dir=os.path.join(XAI_CB_PATH, "tb")      # TensorBoard events here
os.makedirs(base_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

gm_cb = MultiModalGradCAMCallback(
    data_source=validation_ds,
    class_index=0,
    layer_name=None,
    base_dir=XAI_CB_PATH,
    log_dir=XAI_CB_PATH,
    max_samples=8,
    mode="context",
    use_guided_grads=True,
    image_weight=0.8,
    # deprocess_fn=lambda x: np.clip(x.astype(np.float32)/255.0, 0, 1),
    ensure_positive=True,
    positive_threshold=0.5,
)

In [ ]:
XAI_CB_PATH

In [ ]:
# # If you want to control the fixed batch explicitly (including display_images)
# # Build a one-off frozen batch (arrays or tensors)
# batch_inputs, batch_labels = next(iter(validation_ds))
# images = batch_inputs["images"][:8]
# features = batch_inputs["features"][:8]
# # Pretty display panels (model inputs are 0..255 floats → map to [0,1])
# display_images = tf.clip_by_value(images, 0.0, 255.0) / 255.0

# frozen = (images, features, display_images)  # will be used every epoch

# gm_cb = MultiModalGradCAMCallback(
#     data_source=frozen,
#     class_index=0,
#     layer_name=None,
#     base_dir=XAI_PATH,
#     log_dir=XAI_PATH,
#     max_samples=8,                 # safe guard; slices even if you pass larger batch
#     mode="context",
#     use_guided_grads=True,
#     image_weight=0.8,
#     # deprocess_fn not needed because we pass display_images
# )

# history = model.fit(
#     training_ds,
#     epochs=CFG.epochs,
#     callbacks=[lr_cb, ckpt_cb, gm_cb],
#     validation_data=validation_ds,
#     verbose=CFG.verbose,
#     class_weight=class_weights,
# )

In [ ]:
model_fpath = os.path.join(MODEL_PATH, 'best_enhanced_model_test.keras')

ckpt_cb = keras.callbacks.ModelCheckpoint(
    model_fpath,   # Filepath where the model will be saved.
    monitor="val_auc",    # Metric to monitor (validation AUC in this case).
    save_best_only=True,  # Save only the model with the best performance.
    save_weights_only=False,  # Save the entire model (not just the weights).
    mode="max",           # The model with the maximum 'val_auc' will be saved.
)

In [ ]:
cbs = [lr_cb, ckpt_cb, gm_cb]

history = model.fit(
    training_ds,
    epochs=CFG.epochs,
    callbacks=cbs,
    validation_data=validation_ds,
    verbose=CFG.verbose,
    class_weight=class_weights,
)

In [ ]:
# %reload_ext tensorboard
# %tensorboard --logdir "{os.path.join(XAI_PATH, 'tb')}"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# model.save(model_fpath, include_optimizer=True)